# BART Ridership — Kaggle Çalışma Notebook'u

Bu notebook **Kaggle'da çalıştırılır**. Asıl mantık modüler `src/` kodundadır ve
GitHub'dan `clone` ile gelir. Projedeki her adım (Adım 2, 3, 4 ...) **sırayla bu
notebook'a eklenir** — üst üste birikir.

> İş bölümü: `src/` ve yerel scriptler repo'da (yerelde doğrulanır). Bu notebook
> sadece **Kaggle tarafıdır** (clone + import + çalıştır).

## 0. Kurulum — repo clone + import yolu

Notebook ayarları (sağ panel): **Internet → On** (clone için zorunlu),
**Accelerator → GPU** (Adım 4 eğitiminden itibaren; Adım 2 için gerekmez).

In [ ]:
import os, sys

REPO = "bay-area-transit-ridership-forecasting"
URL  = "https://github.com/osman-ozcanli/bay-area-transit-ridership-forecasting.git"

# Repo yoksa clone'la (hucre tekrar calistirilirsa hata vermesin)
if not os.path.exists(REPO):
    os.system(f"git clone {URL}")
sys.path.insert(0, f"/kaggle/working/{REPO}")
print("setup ok ->", REPO)

## Adım 2 — Veri Yükleme (tam veri)

Kaggle'da `use_sample=False` ile **tam ~13.3M satır** yüklenir (yereldeki 1.12M'lik
örnek değil). `/kaggle/input/bart-ridership/` altında 3 CSV ekli olmalı.

`src/data/load.py` içindeki `load_dataset`: 2016+2017 birleştir → DateTime parse →
istasyon merge → WSPR ekle.

In [ ]:
from src.config import load_config
from src.data.load import load_dataset

cfg = load_config()
cfg["environment"] = "kaggle"   # /kaggle/input path'leri
cfg["use_sample"]  = False      # tam veri (~13.3M)

df = load_dataset(cfg)
print("shape:", df.shape)
print("date range:", df["DateTime"].min(), "->", df["DateTime"].max())
df.head(3)

## Adım 3 — Feature Engineering

`src/features/build_features.py`: zaman (Hour/DayOfWeek/Month/IsWeekend/Period) +
tatil (CA, IsHoliday) + mesafe (haversine dist_km) feature'ları, **self-trip drop**
(analiz 3.5), **lag/rolling** (leakage-safe, OD-bazlı) ve LightGBM için **category
dtype** (cat.codes ordinal fix, analiz 3.3).

> Not: Tam veride (~13.3M) groupby/rolling işlemi biraz sürebilir.

In [ ]:
from src.features.build_features import build_features

df = build_features(df, cfg)

print("shape:", df.shape)
lag_cols = [c for c in df.columns if "lag" in c or "roll" in c]
print("lag/rolling kolonlari:", lag_cols)
print("self-trip kalan:", int((df["Origin"] == df["Destination"]).sum()))
print("IsHoliday orani:", round(df["IsHoliday"].mean(), 4))
df.head(3)

## Adım 4 — Temporal Split + LightGBM (GPU) + Model Kayıt

`src/models/train.py`: **2016 train / 2017 test** (temporal split, leakage yok),
**TimeSeriesSplit CV** (varyans), LightGBM **categorical_feature** + **GPU**,
early stopping ve model kayıt (`models/bart_lgb_final.txt`).

> GPU build yoksa otomatik CPU'ya düşer (uyarı basar, çökmeT). CV uzun gelirse
> `config.yaml > model > run_cv: false`.

In [ ]:
from src.models.train import train_model, save_model

cfg["model"]["device"] = "gpu"   # Kaggle GPU

model, results = train_model(df, cfg)

m = results['metrics']
print("device       :", results["device"])
print("train / test :", f"{results['n_train']:,} / {results['n_test']:,}")
print("best_iter    :", results["best_iteration"])
print("holdout MAE  :", round(m["mae"], 4))
print("holdout RMSE :", round(m["rmse"], 4))
print("holdout R2   :", round(m["r2"], 4))
if "cv" in results:
    cv = results['cv']
    print("CV MAE       :", round(cv["mean_mae"], 4), "+/-", round(cv["std_mae"], 4))

path = save_model(model, cfg)
print("model saved  :", path)